In [1]:
#Importação das bibliotecas essenciais
import sys
import os
import pandas as pd
from sqlalchemy import text
from sqlalchemy.types import VARCHAR, FLOAT, DATETIME, INTEGER

sys.path.append("..")
from config import get_engine, PROCESSED_DATA_DIR, get_logger

logger = get_logger("02_carga_mysql")
logger.info("Iniciando processo de carga para o MySQL...")

20:42:59 | INFO     | 02_carga_mysql | Iniciando processo de carga para o MySQL...


In [2]:
# Definindo o caminho do arquivo processado
caminho_parquet = os.path.join(PROCESSED_DATA_DIR, "queimadas_limpo.parquet")
caminho_csv = os.path.join(PROCESSED_DATA_DIR, "queimadas_limpo.csv")

try:
    if os.path.exists(caminho_parquet):
        logger.info(f"Lendo dados processados de: {caminho_parquet}")
        df = pd.read_parquet(caminho_parquet)
    elif os.path.exists(caminho_csv):
        logger.warning(f"Parquet não encontrado. Lendo fallback CSV: {caminho_csv}")
        df = pd.read_csv(caminho_csv)
    else:
        raise FileNotFoundError(
            "Nenhum arquivo processado encontrado. Rode o notebook 01_extracao_e_limpeza antes deste."
        )
    logger.info(f"{len(df):,} registros lidos com sucesso.")
except Exception as e:
    logger.error(f"Falha ao ler dados processados: {e}")
    raise

20:42:59 | INFO     | 02_carga_mysql | Lendo dados processados de: /home/eric/projetos/Analise-Queimadas-Brasil/data/processed/queimadas_limpo.parquet
20:43:00 | INFO     | 02_carga_mysql | 1,011,372 registros lidos com sucesso.


In [3]:
# Criando o banco de dados, usando credenciais do .env via config.py
try:
    engine_server = get_engine(database=None)
    with engine_server.connect() as conn:
        conn.execute(text("CREATE DATABASE IF NOT EXISTS db_queimadas;"))
        conn.commit()
    logger.info("Banco de dados 'db_queimadas' verificado/criado com sucesso.")
except Exception as e:
    logger.error(f"Falha ao criar/verificar o banco de dados: {e}")
    raise

20:43:00 | INFO     | 02_carga_mysql | Banco de dados 'db_queimadas' verificado/criado com sucesso.


In [4]:
# Definindo dtype explícito para a tabela de staging
dtype_staging = {
    "data_hora": DATETIME(),
    "satelite": VARCHAR(50),
    "pais": VARCHAR(50),
    "estado": VARCHAR(50),
    "municipio": VARCHAR(100),
    "bioma": VARCHAR(50),
    "dias_sem_chuva": FLOAT(),
    "precipitacao": FLOAT(),
    "risco_fogo": FLOAT(),
    "frp": FLOAT(),
    "latitude": FLOAT(),
    "longitude": FLOAT(),
    "ano": INTEGER(),
    "mes": INTEGER(),
    "dia": INTEGER(),
}

# Deixando apenas as colunas que existem no DataFrame
dtype_staging = {k: v for k, v in dtype_staging.items() if k in df.columns}


In [5]:
# Enviando os dados para a tabela de staging.
engine_db = get_engine(database="db_queimadas")

logger.info(f"Iniciando a transferência de {len(df):,} registros. Isso pode levar alguns minutos...")
try:
    df.to_sql(
        name="staging_queimadas",
        con=engine_db,
        if_exists="replace",
        index=False,
        chunksize=50000,
        dtype=dtype_staging,
    )
    logger.info("Carga concluída com sucesso.")
except Exception as e:
    logger.error(f"Falha na carga para o MySQL: {e}")
    raise


20:43:00 | INFO     | 02_carga_mysql | Iniciando a transferência de 1,011,372 registros. Isso pode levar alguns minutos...
20:43:34 | INFO     | 02_carga_mysql | Carga concluída com sucesso.


In [6]:
# Garante que nenhuma linha se perdeu no caminho
with engine_db.connect() as conn:
    total_mysql = conn.execute(text("SELECT COUNT(*) FROM staging_queimadas;")).scalar()

if total_mysql != len(df):
    logger.warning(
        f"Divergência de contagem! DataFrame: {len(df):,} | MySQL: {total_mysql:,}. "
        "Investigue antes de seguir para a modelagem dimensional (sql/)."
    )
else:
    logger.info(f"Verificação OK: {total_mysql:,} registros confirmados em 'staging_queimadas'.")


20:43:34 | INFO     | 02_carga_mysql | Verificação OK: 1,011,372 registros confirmados em 'staging_queimadas'.
